# Container terminal lifecycle costs with OpenTESim

This example connects **OpenQTSim service-level simulations** to the **actual OpenTESim cash-flow engine**, following the neighbouring notebook [0031 - Parametric terminal design - Container Terminal (input from dictionaries)](../../OpenTESim/notebooks/0031%20-%20Parametric%20terminal%20design%20-%20Container%20Terminal%20%28input%20from%20dictionaries%29.ipynb). It uses the *Ports and Waterways* container-terminal inputs also used in 3004, but replaces annualised-capital costing with an asset-based, calendar-year lifecycle analysis.

We compare a deliberately small **shortlist**, not every possible layout. Each alternative has a quay, STS crane fleet, tractor-trailer fleet and storage stacks. OpenTESim constructs and aggregates their cash flows. OpenQTSim estimates the waiting/service ratio. We then compare discounted terminal costs and, separately, costs including vessel time among alternatives passing a 10% service screen.

**Dependency:** install the neighbouring checkout into your existing environment with `pip install -e ../OpenTESim` from the OpenQTSim repository root (or `pip install -e ../../OpenTESim` from this notebooks folder). Do not create a new environment. This notebook does not modify either library or require saved outputs from 3004. Moving the notebook does not require the source checkout path if a compatible OpenTESim is already installed.

In [1]:
from copy import deepcopy
from importlib.metadata import version
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import t
from IPython.display import display
import openqtsim
import opentesim
from openqtsim.mt_engine import create_tasks, run_tasks, statistics_worker

print(f'OpenQTSim {openqtsim.__version__}; OpenTESim {version("OpenTESim")}')
print(f'OpenTESim loaded from: {opentesim.__file__}')

OpenQTSim v0.5.1; OpenTESim 0.0.0
OpenTESim loaded from: /Users/mvankoningsveld/Library/CloudStorage/OneDrive-DelftUniversityofTechnology/github/OpenTESim/src/opentesim/__init__.py


## 1. Inputs and financial conventions

Operational inputs follow *Ports and Waterways*, Part II Chapter 4 Section 4.4. Financial templates follow OpenTESim 0031: quay life 30 years, STS crane life 40 years, trailer/stack life 50 years; quay construction shares 20/40/40%; 3% annual variable costs plus insurance; 8% decommissioning; five trailers per crane. The empirical quay cost is `753.24 * retaining_height**1.2729` per metre plus mobilisation, as in 0031. Those are historical/illustrative assumptions, not current quotes.

**Common currency is USD**, consistent with the source's quay-cost formula; equipment prices are assumed to be in the same currency and price basis. No EUR/USD conversion or contemporary price adjustment is implied. These numbers are not directly comparable with the illustrative EUR costs in 3004.

The default uses **constant prices and zero escalation** with a **7% real discount rate**. This is a deliberate simplification relative to 0031's mixed escalation settings: do not combine nominal escalation with a real discount rate. OpenTESim's default project `npv` discounts base flows; here we explicitly discount the reconciled constant-price net cash flow. The installed engine's project rebuild does not preserve a complete escalated component ledger, so its escalated totals are **not used** for financial decisions or plots. Depreciation determines engine residual value; it is not an additional cash expense or tax model.

Costs are negative, optional handling revenue positive. `tariff_per_teu=0` makes the default a **cost-only** assessment, not a profitability claim. Vessel waiting/service costs are external to the terminal operator and are added only in a separate economic view.

In [2]:
throughput_teu = 2_460_000
peak_factor = 1.2
operating_hours = 8_592
crane_productivity = 30 * 2  # TEU/hour/crane
mooring_overhead = 2.0  # hours/call, mooring + unmooring
max_waiting_factor = 0.10
candidates = [(3, 12), (4, 12), (4, 14), (4, 16), (5, 16)]
fleet = pd.DataFrame({'class': ['Post Panamax I', 'VLCS', 'ULCS'],
                      'TEU share': [0.4, 0.3, 0.3], 'call TEU': [1750, 3250, 4875],
                      'length m': [300, 397, 400]}).set_index('class')
nr_arr = 30_000
nr_replications = 4
seed = 2024
client = None  # optionally supply an existing Dask Client

start_year = 2023
project_years = 50  # includes the three construction years
discount_base_year = start_year
discount_rate = 0.07  # real; all calculations below use constant prices
retaining_height = 23.0  # m
mobilisation = 2_500_000.0  # USD
tariff_per_teu = 0.0  # optional USD/TEU terminal revenue, NOT from the book
vessel_cost_per_hour = 1_500.0  # USD/vessel-hour, illustrative external cost
quay_template = dict(construction_duration=3, share_of_investments=[0.2, 0.4, 0.4],
                     economic_lifetime=30, yearly_variable_costs_rate=0.03, insurance_rate=0.005)
crane_template = dict(capex_unit=10_000_000.0, construction_duration=1, share_of_investments=[1.0],
                      economic_lifetime=40, yearly_variable_costs_rate=0.03, insurance_rate=0.01)
trailer_template = dict(capex_unit=250_000.0, construction_duration=1, share_of_investments=[1.0],
                        economic_lifetime=50, yearly_variable_costs_rate=0.03, insurance_rate=0.01)
stack_template = dict(capex_unit=1_000_000.0, construction_duration=1, share_of_investments=[1.0],
                      economic_lifetime=50, yearly_variable_costs_rate=0.03, insurance_rate=0.01)
decommissioning_rate = 0.08
trailers_per_crane = 5

## 2. Size the physical asset groups

Fleet shares are TEU shares, so round each class's flow/call-size upward to obtain annual calls; use call shares for mean vessel length. Quay length uses the same largest-vessel plus average-vessel rule as 3004.

Storage follows the source's four cargo classes and discrete stack sizing: terminal-side throughput is quay throughput times `(0.15 + 0.16 + 0.69/2)` because transhipment crosses the quay twice but occupies the stack once. Peak factor, dwell times, stacking height, occupancy and stack dimensions determine stack counts. These stacks are costed as assets; no yard queue is simulated. The source's simplified fixed price per stack is retained, despite differing stack dimensions.

Scope is still **partial terminal infrastructure**, not a complete bankable terminal model: gates, buildings, land, dredging, labour beyond template OPEX, power consumption, taxes and financing schedules are not independently sized/costed. Do not infer a complete project profit from this shortlist.

In [3]:
if not np.isclose(fleet['TEU share'].sum(), 1) or (fleet <= 0).any().any():
    raise ValueError('Supply positive fleet inputs and TEU shares summing to one.')
if not candidates or len(set(candidates)) != len(candidates):
    raise ValueError('Supply a nonempty shortlist without duplicate designs.')
if any(not isinstance(value, int) or value < 1 for pair in candidates for value in pair):
    raise ValueError('Berth and crane counts must be positive integers.')
if any(not n <= k <= 4 * n for n, k in candidates):
    raise ValueError('Each design must provide between one and four cranes per berth on average.')
if nr_replications < 2 or nr_arr < 1 or project_years <= quay_template['construction_duration']:
    raise ValueError('Use positive run lengths, at least two replications and an operating project horizon.')
positive = [throughput_teu, peak_factor, operating_hours, crane_productivity, retaining_height, max_waiting_factor]
if not all(np.isfinite(value) and value > 0 for value in positive):
    raise ValueError('Demand, productivity, annual hours, retaining height and target must be positive and finite.')
nonnegative = [discount_rate, mooring_overhead, mobilisation, tariff_per_teu, vessel_cost_per_hour, decommissioning_rate]
if not all(np.isfinite(value) and value >= 0 for value in nonnegative):
    raise ValueError('Financial rates, mooring time, revenue and cost inputs must be nonnegative and finite.')
fleet['annual calls'] = np.ceil(throughput_teu * fleet['TEU share'] / fleet['call TEU']).astype(int)
annual_calls = int(fleet['annual calls'].sum())
average_length = float(np.average(fleet['length m'], weights=fleet['annual calls']))
largest_length = float(fleet['length m'].max())
average_exchange = throughput_teu / annual_calls
peak_arrival_rate = peak_factor * annual_calls / operating_hours
operational_start = start_year + quay_template['construction_duration']
end_year = start_year + project_years - 1
cargo = pd.DataFrame({
    'class': ['Laden', 'Reefer', 'Empty', 'OOG'], 'share': [0.70, 0.10, 0.19, 0.01],
    'dwell days': [7.5, 6.5, 11, 7], 'height': [4, 4, 6, 1],
    'stacking fraction': [0.8, 0.8, 1, 1], 'occupancy': [0.7, 0.7, 0.8, 0.8],
    'TEU per stack': [45 * 20 * 4, 22 * 4 * 4, 35 * 24 * 6, 10 * 10],
}).set_index('class')
terminal_throughput = throughput_teu * (0.15 + 0.16 + 0.69 / 2)
cargo['required TEU capacity'] = (terminal_throughput * cargo['share'] * peak_factor * cargo['dwell days']
                                / (operating_hours / 24) / cargo['stacking fraction'] / cargo['occupancy'])
cargo['stacks'] = np.ceil(cargo['required TEU capacity'] / cargo['TEU per stack']).astype(int)
rows = []
for n, k in candidates:
    service = average_exchange / (crane_productivity * k / n) + mooring_overhead
    rows.append({'berths': n, 'cranes': k, 'service hours': service,
                 'occupancy': peak_arrival_rate * service / n,
                 'quay length m': 30 + largest_length + 1.1 * (n - 1) * (15 + average_length)})
designs = pd.DataFrame(rows).set_index(['berths', 'cranes'])
designs['stable'] = designs['occupancy'] < 1
if not designs['stable'].any():
    raise ValueError('No stable design: expand the shortlist or reduce traffic.')
display(fleet)
display(cargo.round(3))
display(designs.round(4))

,TEU share,call TEU,length m,annual calls
class,,,,
Post Panamax I,0.4,1750,300,563
VLCS,0.3,3250,397,228
ULCS,0.3,4875,400,152


,share,dwell days,height,stacking fraction,occupancy,TEU per stack,required TEU capacity,stacks
class,,,,,,,,
Laden,0.70,7.5,4,0.8,0.7,3600,50634.427,15
Reefer,0.10,6.5,4,0.8,0.7,352,6269.024,18
Empty,0.19,11.0,6,1.0,0.8,5040,14110.127,3
OOG,0.01,7.0,1,1.0,0.8,100,472.588,5


service hours  occupancy  quay length m  stable
berths cranes                                                 
3      12            12.8696     0.5650      1210.0575    True
4      12            16.4928     0.5430      1600.0862    True
       14            14.4224     0.4749      1600.0862    True
       16            12.8696     0.4237      1600.0862    True
5      16            15.5870     0.4106      1990.1150    True

## 3. Estimate service quality with OpenQTSim

As in 3004's **extension**, each berth has an effective average k/n cranes, not an integer crane-resource allocation. The model uses E2 arrivals and aggregate E2 service times with the calculated mean; it does not sample individual vessel classes or enforce quay length. The peak factor applies once, to arrival rate. This workload convention differs from the book/OpenTESim 0031's sizing occupancy; it does not reproduce the book's 4/14 decision criterion.

Each design has `nr_replications` independent simulations of `nr_arr` customers. A master seed supplies separate arrival and service seeds per task. The service screen requires an approximate one-sided 95% Student-t upper bound on the mean waiting/service factor to be below the limit. This is not a simultaneous guarantee over the shortlist and excludes start-up bias and model/parameter uncertainty. All runs start empty; rerun with longer runs and more replications around close decisions.

In [4]:
stable_designs = designs[designs['stable']]
children = np.random.SeedSequence(seed).spawn(len(stable_designs))
tasks, task_designs = [], []
for ((n, k), design), child in zip(stable_designs.iterrows(), children):
    batch = create_tasks('E2', 'E2', [n], [design['occupancy']], nr_arr,
                         mu=n / design['service hours'], nr_replications=nr_replications,
                         seed=int(child.generate_state(1)[0]))
    tasks.extend(batch)
    task_designs.extend([(n, k)] * len(batch))
results = pd.DataFrame(run_tasks(tasks, client, func=statistics_worker))
results['berths'] = [n for n, _ in task_designs]
results['cranes'] = [k for _, k in task_designs]
summary = results.groupby(['berths', 'cranes'])['W_q'].agg(['mean', 'std', 'count'])
designs['wait hours'] = summary['mean']
designs['wait SE hours'] = summary['std'] / np.sqrt(summary['count'])
designs['waiting factor'] = designs['wait hours'] / designs['service hours']
designs['factor upper bound'] = (designs['waiting factor']
                                + t.ppf(0.95, nr_replications - 1) * designs['wait SE hours'] / designs['service hours'])
designs['passes screening'] = designs['stable'] & (designs['factor upper bound'] <= max_waiting_factor)
if not designs['passes screening'].any():
    raise ValueError('No candidate passes the service screen. Expand capacity or investigate uncertainty; do not choose a failing design.')
display(designs[['occupancy', 'waiting factor', 'factor upper bound', 'passes screening']].round(4))

occupancy  waiting factor  factor upper bound  passes screening
berths cranes                                                                 
3      12         0.5650          0.0862              0.0894              True
4      12         0.5430          0.0406              0.0444              True
       14         0.4749          0.0194              0.0201              True
       16         0.4237          0.0115              0.0118              True
5      16         0.4106          0.0046              0.0052              True

## 4. Build genuine OpenTESim asset objects

The same mixins and registry keys as in 0031 are used. To keep execution quick, identical cranes/trailers/stacks are grouped into a **fleet asset** with quantity times unit CAPEX. For common timing, rates and lifetime this is cash-flow equivalent to individual assets; the quantities remain in an explicit inventory. Storage classes remain separate. No source files are copied and no cash-flow engine is reimplemented.

The quay starts construction in 2023 and all asset groups enter service in 2026. The 50-year project horizon includes construction, so there are 47 operating years, not 50. `reinvestment_policy='overlap'` schedules replacement construction before the previous asset retires, maintaining aggregate availability. The engine handles final residual value and decommissioning according to its project-end conventions; this is not a detailed demolition/construction/downtime plan.

In [5]:
AggregationElement = type('AggregationElement', (opentesim.core.isIdentifiable, opentesim.core.hasHierarchy), {})
SystemElement = type('SystemElement', (
    opentesim.core.isIdentifiable, opentesim.core.hasType,
    opentesim.cashflow.hasCashflowMixin, opentesim.cashflow.hasCashflowTemplate,
    opentesim.cashflow.hasCashflowBuilder), {})
CLASS_REGISTRY = {'AggregationElement': AggregationElement, 'SystemElement': SystemElement}

def create_asset(name, asset_type, quantity, unit_capex, template):
    """Create an OpenTESim fleet asset using the source notebook's template conventions."""
    data = deepcopy(template)
    data.pop('capex_unit', None)
    data.update(name=name, type=asset_type,
                investment_start_year=operational_start - data['construction_duration'],
                capex_investment=-quantity * unit_capex,
                escalation_base_year=start_year, escalation_rate=0.0,
                depreciation_rate=1 / data['economic_lifetime'],
                decommissioning_rate=decommissioning_rate)
    asset = SystemElement(name=name, type=asset_type)
    asset.setup_cashflow_axis(investment_start_year=data['investment_start_year'],
                              construction_duration=data['construction_duration'], economic_lifetime=data['economic_lifetime'])
    asset.add_series_from_template(data, add_escalated=False,
                                   include_decommissioning=True, include_revenue=False)
    return asset

def build_terminal(n, k, design):
    """Return a fresh OpenTESim hierarchy and auditable asset inventory for one alternative."""
    terminal = AggregationElement(name=f'{n} berths / {k} cranes')
    terminal.children = []
    quay_capex = 753.24 * retaining_height ** 1.2729 * design['quay length m'] + mobilisation
    specifications = [
        ('Quay', 'Quay', 1, quay_capex, quay_template),
        ('STS fleet', 'STS Crane', k, crane_template['capex_unit'], crane_template),
        ('Tractor-trailer fleet', 'Tractor trailer', k * trailers_per_crane, trailer_template['capex_unit'], trailer_template),
    ]
    specifications.extend((f'{name} stacks', 'Storage stack', int(row['stacks']),
                           stack_template['capex_unit'], stack_template) for name, row in cargo.iterrows())
    inventory = []
    for name, asset_type, quantity, unit_capex, template in specifications:
        terminal.children.append(create_asset(name, asset_type, quantity, unit_capex, template))
        inventory.append({'asset': name, 'type': asset_type, 'quantity': quantity,
                          'initial CAPEX USD': quantity * unit_capex,
                          'operational start': operational_start, 'life years': template['economic_lifetime']})
    return terminal, pd.DataFrame(inventory).set_index('asset')

## 5. Calculate and reconcile calendar-year cash flows

`loop_calculate_cashflows` performs actual asset replacement, horizon alignment and aggregation. Keep the canonical **total_capex**, **total_opex** and **total_revenue** columns; never sum both detailed components and their totals (or base and escalated versions), which would double-count.

Optional handling revenue is a separate project-level series, charged once per quay TEU during operating years. It is not attached to cranes or replacements, avoiding repeated revenue. Vessel costs are also separate: nominal annual calls times peak-demand estimated waiting plus service time. As in 3004, that is a **peak-delay annualisation**, not an annual seasonal simulation. Annual vessel costs stop at the project end. They are not terminal OPEX unless the operator actually pays them.

For each year y, discount factor is `(1 + rate)**(y - discount_base_year)`; 2023 is t=0. This follows 0031's year-index convention rather than imposing an extra end-of-year offset. Terminal NPV is revenue minus lifecycle terminal costs; extended NPV additionally subtracts vessel time. Maximise the relevant NPV **only among service-qualified candidates**. A less-negative cost-only NPV is preferable, but is not a profitable project.

In [6]:
projects, inventories, engine_cashflows, cashflows = {}, {}, {}, {}
for (n, k), design in designs[designs['stable']].iterrows():
    terminal, inventory = build_terminal(n, k, design)
    terminal, engine = opentesim.cashflow.loop_calculate_cashflows(
        terminal, CLASS_REGISTRY, start_year=start_year, end_year=end_year,
        npv_base_year=discount_base_year, WACC=discount_rate,
        reinvest=True, escalation=0.0, reinvestment_policy='overlap',
        default_decommissioning_rate=decommissioning_rate)
    np.testing.assert_allclose(engine['net_cashflow'],
                               engine['total_capex'] + engine['total_opex'] + engine['total_revenue'])
    frame = engine[['total_capex', 'total_opex', 'total_revenue']].copy()
    active = frame.index >= operational_start
    frame['handling revenue'] = np.where(active, throughput_teu * tariff_per_teu, 0.0)
    frame['vessel time cost'] = np.where(active, -annual_calls * vessel_cost_per_hour * (design['wait hours'] + design['service hours']), 0.0)
    frame['operator net'] = frame['total_capex'] + frame['total_opex'] + frame['total_revenue'] + frame['handling revenue']
    frame['extended net'] = frame['operator net'] + frame['vessel time cost']
    frame['discount factor'] = (1 + discount_rate) ** (frame.index.to_numpy() - discount_base_year)
    frame['operator discounted'] = frame['operator net'] / frame['discount factor']
    frame['extended discounted'] = frame['extended net'] / frame['discount factor']
    frame['operator cumulative NPV'] = frame['operator discounted'].cumsum()
    frame['extended cumulative NPV'] = frame['extended discounted'].cumsum()
    pv_terminal_cost = float((-(frame['total_capex'] + frame['total_opex']) / frame['discount factor']).sum())
    pv_teu = float(np.where(active, throughput_teu / frame['discount factor'], 0.0).sum())
    designs.loc[(n, k), 'PV terminal cost USD'] = pv_terminal_cost
    designs.loc[(n, k), 'PV vessel time USD'] = float((-frame['vessel time cost'] / frame['discount factor']).sum())
    designs.loc[(n, k), 'operator NPV USD'] = float(frame['operator discounted'].sum())
    designs.loc[(n, k), 'extended NPV USD'] = float(frame['extended discounted'].sum())
    designs.loc[(n, k), 'break-even tariff USD/TEU'] = pv_terminal_cost / pv_teu
    if tariff_per_teu == 0:
        np.testing.assert_allclose(frame['operator discounted'], engine['npv'])
    projects[(n, k)], inventories[(n, k)] = terminal, inventory
    engine_cashflows[(n, k)], cashflows[(n, k)] = engine, frame

qualified = designs[designs['passes screening']]
operator_choice = qualified['operator NPV USD'].idxmax()
economic_choice = qualified['extended NPV USD'].idxmax()
display(designs[['waiting factor', 'factor upper bound', 'passes screening',
                 'PV terminal cost USD', 'PV vessel time USD', 'operator NPV USD',
                 'extended NPV USD', 'break-even tariff USD/TEU']].round(4))
print(f'Terminal-operator choice: {operator_choice}; wider economic choice: {economic_choice}')
print('Choices are estimates over this shortlist, not global optima.')

Terminal-operator choice: (np.int64(3), np.int64(12)); wider economic choice: (np.int64(3), np.int64(12))
Choices are estimates over this shortlist, not global optima.


waiting factor  ...  break-even tariff USD/TEU
berths cranes                  ...                           
3      12              0.0862  ...                    10.7710
4      12              0.0406  ...                    11.5547
       14              0.0194  ...                    12.6099
       16              0.0115  ...                    13.6651
5      16              0.0046  ...                    14.4489

[5 rows x 8 columns]

## 6. Inspect timing, replacement and project-end value

The selected economic alternative is shown below. Inspect 2023-2025 construction spending and 2026 operations. With the source lifetimes and overlap policy, quay replacement construction starts in **2053**, crane replacement in **2065**. Trailers/stacks last beyond the horizon. These years follow the inputs and are recomputed after edits.

At the final year the engine posts residual value (a positive capital cash flow) and decommissioning (negative OPEX) for last-generation assets. It suppresses disposal costs for intermediate replacement generations under its current policy. That is an engine convention, not a detailed site-clearance estimate. The ledger exposes the postings so they can be reviewed. No separate capital recovery annuity is charged on top of construction CAPEX.

Break-even tariff equals PV of terminal lifecycle net costs divided by discounted handled TEU, **excluding vessel costs**, with full constant throughput from opening and zero taxes. It is a simplified infrastructure charge for the included scope, not a complete commercial tariff.

In [7]:
selected = economic_choice
selected_engine = engine_cashflows[selected]
selected_cashflow = cashflows[selected]
display(inventories[selected].round(2))
audit_columns = ['capex_construction', 'capex_divestment', 'opex_variable_costs',
                 'opex_insurance', 'opex_decommissioning']
audit_years = sorted(set(selected_engine.index[selected_engine['capex_construction'] != 0])
                     | {operational_start, end_year})
display(selected_engine.loc[audit_years, audit_columns].round(2))
display(selected_cashflow[['operator net', 'vessel time cost', 'extended net',
                           'operator cumulative NPV', 'extended cumulative NPV']].round(2))

,type,quantity,initial CAPEX USD,operational start,life years
asset,,,,,
Quay,Quay,1,5.182678e+07,2026,30
STS fleet,STS Crane,12,1.200000e+08,2026,40
Tractor-trailer fleet,Tractor trailer,60,1.500000e+07,2026,50
Laden stacks,Storage stack,15,1.500000e+07,2026,50
Reefer stacks,Storage stack,18,1.800000e+07,2026,50
Empty stacks,Storage stack,3,3.000000e+06,2026,50
OOG stacks,Storage stack,5,5.000000e+06,2026,50


,capex_construction,capex_divestment,opex_variable_costs,opex_insurance,opex_decommissioning
years,,,,,
2023,-1.036536e+07,0.000000e+00,0.0,0.00,0.00
2024,-2.073071e+07,0.000000e+00,0.0,0.00,0.00
2025,-1.967307e+08,0.000000e+00,0.0,0.00,0.00
2026,0.000000e+00,0.000000e+00,-6834803.3,-2019133.88,0.00
2053,-1.036536e+07,0.000000e+00,-6834803.3,-2019133.88,0.00
2054,-2.073071e+07,0.000000e+00,-6834803.3,-2019133.88,0.00
2055,-2.073071e+07,0.000000e+00,-6834803.3,-2019133.88,0.00
2065,-1.200000e+08,0.000000e+00,-6834803.3,-2019133.88,0.00
2072,0.000000e+00,1.248183e+08,-6834803.3,-2019133.88,-18226142.13


,operator net,vessel time cost,extended net,operator cumulative NPV,extended cumulative NPV
years,,,,,
2023,-1.036536e+07,0.00,-1.036536e+07,-1.036536e+07,-1.036536e+07
2024,-2.073071e+07,0.00,-2.073071e+07,-2.973985e+07,-2.973985e+07
2025,-1.967307e+08,0.00,-1.967307e+08,-2.015721e+08,-2.015721e+08
2026,-8.853937e+06,-19772500.11,-2.862644e+07,-2.087995e+08,-2.249398e+08
2027,-8.853937e+06,-19772500.11,-2.862644e+07,-2.155541e+08,-2.467787e+08
2028,-8.853937e+06,-19772500.11,-2.862644e+07,-2.218669e+08,-2.671890e+08
2029,-8.853937e+06,-19772500.11,-2.862644e+07,-2.277666e+08,-2.862640e+08
2030,-8.853937e+06,-19772500.11,-2.862644e+07,-2.332804e+08,-3.040911e+08
2031,-8.853937e+06,-19772500.11,-2.862644e+07,-2.384335e+08,-3.207520e+08


### Preserve the OpenTESim ledger
Keep asset, replacement generation, component and year detail for both plots. Missing component columns mean that generation has no such posting (for example, intermediate generations have no decommissioning posting). Reconcile every component and year to the engine totals. Identical cranes, trailers and stacks remain the quantity-weighted fleets used in the model; no individual assets are invented.

In [8]:
selected_inventory = inventories[selected]
asset_groups = {name: ('Quay' if name == 'Quay' else
                       'Storage' if row['type'] == 'Storage stack' else 'Waterside equipment')
                for name, row in selected_inventory.iterrows()}
component_columns = [column for column in selected_engine.columns
                     if column.startswith(('capex_', 'opex_', 'revenue_'))
                     and not column.endswith(('_escalated', '_discounted', '_summed'))]

def generation_cashflows(node, path=()):
    """Keep the full engine hierarchy, yielding only leaf-generation ledgers."""
    path = (*path, node.name)
    if getattr(node, 'children', []):
        for child in node.children:
            yield from generation_cashflows(child, path)
    else:
        yield path, node.df

generation_ledgers, detail_rows = {}, []
for asset in projects[selected].children:
    for path, ledger in generation_cashflows(asset):
        # Absent components are not posted for that generation (e.g. intermediate disposal).
        aligned = ledger.reindex(index=selected_engine.index, columns=component_columns, fill_value=0)
        generation_ledgers[path] = aligned
        generation = ' / '.join(path[1:]) if len(path) > 1 else 'Initial generation'
        for component in component_columns:
            for year, value in aligned[component].items():
                detail_rows.append({'group': asset_groups[asset.name], 'asset': asset.name,
                                    'generation': generation, 'component': component, 'year': year,
                                    'cash flow USD': float(value),
                                    'PV USD': float(value / selected_cashflow.loc[year, 'discount factor'])})
cashflow_detail = pd.DataFrame(detail_rows)
np.testing.assert_allclose(sum(generation_ledgers.values()), selected_engine[component_columns], atol=1e-6)
np.testing.assert_allclose(cashflow_detail.groupby('year')['cash flow USD'].sum(),
                           selected_engine['net_cashflow'], atol=1e-6)


## 7. Visualise annual cash flows and compare perspectives

The first figure retains every posted OpenTESim asset, replacement generation and cash-flow component as a separate signed annual series. Hover for detail and use the legend to isolate series. Residual credits and decommissioning are not netted into construction or maintenance. Discounted cumulative values use a separate axis; project handling revenue and external vessel costs remain separate. The second compares the present-value cost mix and service target across stable candidates, including those failing the screen.

In [9]:
cashflow_fig = make_subplots(specs=[[{'secondary_y': True}]])
for (asset, generation, component), detail in cashflow_detail.groupby(['asset', 'generation', 'component'], sort=False):
    if detail['cash flow USD'].ne(0).any():
        cashflow_fig.add_bar(x=detail['year'], y=detail['cash flow USD'] / 1e6,
                             name=f'{asset} / {generation} / {component}', legendgroup=asset,
                             hovertemplate='%{fullData.name}<br>Year %{x}<br>Million USD %{y:,.4f}<extra></extra>',
                             secondary_y=False)
for column, label in [('handling revenue', 'Handling revenue'), ('vessel time cost', 'External vessel time')]:
    cashflow_fig.add_bar(x=selected_cashflow.index, y=selected_cashflow[column] / 1e6, name=label, secondary_y=False)
for column, label in [('operator cumulative NPV', 'Cumulative operator NPV'), ('extended cumulative NPV', 'Cumulative extended NPV')]:
    cashflow_fig.add_scatter(x=selected_cashflow.index, y=selected_cashflow[column] / 1e6, name=label,
                             mode='lines', secondary_y=True)
cashflow_fig.update_layout(title=f'OpenTESim asset cash flows: {selected[0]} berths / {selected[1]} cranes',
                           barmode='relative', template='plotly_white', height=600)
cashflow_fig.update_yaxes(title_text='Annual constant-price cash flow (million USD)', secondary_y=False)
cashflow_fig.update_yaxes(title_text='Cumulative discounted net value (million USD)', secondary_y=True)
annual_bars = np.array([trace.y for trace in cashflow_fig.data if trace.type == 'bar'])
annual_extent = max(np.maximum(annual_bars, 0).sum(axis=0).max(),
                    -np.minimum(annual_bars, 0).sum(axis=0).min())
npv_extent = selected_cashflow[['operator cumulative NPV', 'extended cumulative NPV']].abs().to_numpy().max() / 1e6
cashflow_fig.update_yaxes(range=[-1.05 * max(annual_extent, 1), 1.05 * max(annual_extent, 1)], secondary_y=False)
cashflow_fig.update_yaxes(range=[-1.05 * max(npv_extent, 1), 1.05 * max(npv_extent, 1)], secondary_y=True)
cashflow_fig.update_xaxes(title_text='Calendar year')
cashflow_fig.show()

comparison = designs[designs['stable']]
labels = [f'{n} berths / {k} cranes' for n, k in comparison.index]
comparison_fig = make_subplots(rows=1, cols=2, subplot_titles=('Present-value cost mix', 'Mean waiting/service ratio'))
for column in ['PV terminal cost USD', 'PV vessel time USD']:
    comparison_fig.add_bar(x=labels, y=comparison[column] / 1e6, name=column, row=1, col=1)
comparison_fig.add_scatter(x=labels, y=comparison['waiting factor'], mode='markers',
                           marker=dict(color=['green' if value else 'red' for value in comparison['passes screening']], size=10),
                           name='Service estimate (green passes)', error_y=dict(type='data', symmetric=False,
                           array=comparison['factor upper bound'] - comparison['waiting factor'], arrayminus=np.zeros(len(comparison))), row=1, col=2)
comparison_fig.add_hline(y=max_waiting_factor, line_dash='dash', row=1, col=2)
comparison_fig.update_layout(title='Service-qualified lifecycle alternatives', barmode='stack', template='plotly_white', height=600)
comparison_fig.update_yaxes(title_text='Present value (million USD)', row=1, col=1)
comparison_fig.update_yaxes(title_text='E[waiting] / E[service]', row=1, col=2)
comparison_fig.show()

### Initial investment breakdown

For `selected` (the wider-economic choice above), this sunburst shows **undiscounted initial CAPEX in USD**, grouped into quay, waterside equipment and storage. Replacement investments, operating costs and residual value are excluded. Fleet and stack values already include their quantities. Click a sector to explore its breakdown.

In [10]:
selected_inventory = inventories[selected]
asset_groups = {name: ('Quay' if name == 'Quay' else
                       'Storage' if row['type'] == 'Storage stack' else 'Waterside equipment')
                for name, row in selected_inventory.iterrows()}
investment_ids, investment_labels, investment_parents, investment_values = ['terminal'], ['Terminal'], [''], [selected_inventory['initial CAPEX USD'].sum()]
for group in dict.fromkeys(asset_groups.values()):
    members = [name for name, parent in asset_groups.items() if parent == group]
    investment_ids.append(f'group/{group}')
    investment_labels.append(group)
    investment_parents.append('terminal')
    investment_values.append(selected_inventory.loc[members, 'initial CAPEX USD'].sum())
    for name in members:
        investment_ids.append(f'asset/{name}')
        investment_labels.append(name)
        investment_parents.append(f'group/{group}')
        investment_values.append(selected_inventory.loc[name, 'initial CAPEX USD'])
investment_sunburst = opentesim.plot.create_sunburst_plot(
    investment_labels, investment_parents, investment_values,
    title=f'Initial investment: {selected[0]} berths / {selected[1]} cranes',
    hovertemplate='%{label}<br>USD %{value:,.0f}<br>%{percentRoot:.1%} of total<extra></extra>')
investment_sunburst.update_traces(ids=investment_ids)
investment_sunburst.show()


### Lifecycle cost breakdown

This sunburst preserves **asset group, asset, replacement generation, cash-flow component and calendar year**, using the same detailed engine ledger as the cash-flow plot. Click sectors to drill down. Values are discounted to the NPV base year. Only leaf generations are summed, avoiding double counting aggregation parents.

The root shows the **magnitude of all terminal cash flows**, not net cost: cost outflows and credits have separate branches because sunburst areas cannot be negative. Residual value is retained in the credit branch with the same asset/generation/component/year detail; hover reports signed PV. The reconciliation table subtracts credits from gross costs. Handling revenue and external vessel-time costs remain outside this terminal-asset breakdown.

In [11]:
posted_detail = cashflow_detail[cashflow_detail['cash flow USD'].ne(0)].copy()
posted_detail['direction'] = np.where(posted_detail['PV USD'] < 0, 'Cost outflows', 'Credits')
gross_lifecycle_cost = float(-posted_detail.loc[posted_detail['PV USD'] < 0, 'PV USD'].sum())
terminal_credit_pv = float(posted_detail.loc[posted_detail['PV USD'] > 0, 'PV USD'].sum())
residual_credit_pv = float(posted_detail.loc[posted_detail['component'] == 'capex_divestment', 'PV USD'].sum())
np.testing.assert_allclose(gross_lifecycle_cost - terminal_credit_pv,
                           designs.loc[selected, 'PV terminal cost USD'])
lifecycle_reconciliation = pd.Series({
    'Gross discounted terminal costs': gross_lifecycle_cost,
    'Less: discounted terminal credits (including residual)': terminal_credit_pv,
    'Net discounted terminal cost': gross_lifecycle_cost - terminal_credit_pv}, name='USD')
display(lifecycle_reconciliation.to_frame().round(2))
lifecycle_ids, lifecycle_labels, lifecycle_parents = ['terminal'], ['Terminal cash-flow magnitudes'], ['']
lifecycle_values = [gross_lifecycle_cost + terminal_credit_pv]
lifecycle_signed = [terminal_credit_pv - gross_lifecycle_cost]
hierarchy_columns = ['direction', 'group', 'asset', 'generation', 'component', 'year']
node_ids = {(): 'terminal'}
for depth in range(1, len(hierarchy_columns) + 1):
    for keys, detail in posted_detail.groupby(hierarchy_columns[:depth], sort=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        node_id = f'node/{len(lifecycle_ids)}'
        node_ids[keys] = node_id
        lifecycle_ids.append(node_id)
        lifecycle_labels.append(str(keys[-1]))
        lifecycle_parents.append(node_ids[keys[:-1]])
        lifecycle_values.append(float(detail['PV USD'].abs().sum()))
        lifecycle_signed.append(float(detail['PV USD'].sum()))
lifecycle_sunburst = opentesim.plot.create_sunburst_plot(
    lifecycle_labels, lifecycle_parents, lifecycle_values,
    title=f'Lifecycle costs and credits (PV): {selected[0]} berths / {selected[1]} cranes',
    hovertemplate='%{label}<br>Magnitude PV USD %{value:,.0f}<br>Signed PV USD %{customdata:,.0f}<extra></extra>')
lifecycle_sunburst.update_traces(ids=lifecycle_ids, customdata=lifecycle_signed, maxdepth=4)
lifecycle_sunburst.show()


,USD
Gross discounted terminal costs,3.214013e+08
Less: discounted terminal credits (including residual),4.533912e+06
Net discounted terminal cost,3.168674e+08


## 8. Interpretation and experiments

* Compare operator and wider economic choices: the former excludes vessel delay costs; the latter includes them. Both must pass the service screen. With zero tariff, operator NPV is simply negative PV terminal costs.
* Change discount rate, asset lifetimes and construction shares to see why lifecycle ranking can differ from the annualised-capital model in 3004. Inspect replacement years and final residual assumptions, rather than only the headline NPV.
* Add a nonzero tariff to explore cash inflows. Because all layouts handle identical throughput, a common tariff does not change their operator cost ranking. It can change whether NPV is positive. No cargo growth or ramp-up is assumed.
* The peak waiting-time estimate is applied to nominal annual calls; it is not an observed annual average. The 95% service bound concerns simulation sampling only, not demand uncertainty, cost uncertainty or project viability.
* Try a stricter service limit and extend `candidates`. The shortlist is intentionally small; a boundary solution is not proof of a globally optimal design. Re-simulate close alternatives with longer independent replications before comparing them.

**Traceability:** OpenTESim notebook 0031 supplies the asset-template and lifecycle workflow, equipment prices, quay empirical relation and yard-sizing conventions. *Ports and Waterways*, supplied 2026 PDF, Part II Chapter 4 Section 4.4, supplies the terminal operating example. Added assumptions here include the shortlist, consistent constant-price basis, tariff option, external vessel-time cost, queue simulation convention and statistical service screening. Costs are illustrative. The source notebook and libraries are left unchanged.